# Consistent Face 3x3 Generator — ComfyUI

3x3 grid referans görselden 9 tutarılı yüz görseli üret.

## Pipeline
```
3x3 Grid Image → Upscale (3x) → Crop (9 parça) → Depth LoRA + FLUX.1 Dev → FaceDetailer + EyesDetailer → 9x Consistent Output
```

## Main Model
**FLUX.1 Dev** + **FLUX.1 Fill Dev** (FP8)

## Colab Secrets
- `CF_TUNNEL_TOKEN` — Cloudflare tunnel
- `HF_TOKEN` — HuggingFace (gated model erişimi için gerekli!)

## Önemli Not
- FLUX.1 Dev, Fill, Depth LoRA **gated model** — önce HuggingFace'te lisansı kabul et
- `chinfixer-2000` ve `Pandora-RAWr` LoRA'ları **CivitAI**'dan manuel indirilmeli

## Kullanım
A: Kurulum + Node'lar → B: Model indir → C: Başlat

## Custom Node'lar
| Node | Kullanım |
|------|----------|
| ComfyUI-Impact-Pack | FaceDetailer, UltralyticsDetectorProvider |
| ComfyUI-Impact-Subpack | Impact ek |
| rgthree-comfy | Bookmark, workflow utility |
| ComfyUI-KJNodes | Image/mask utility |
| comfyui-depthanythingv2 | Depth Anything V2 |
| ComfyUI_essentials | GetImageSize+ vb. |
| ComfyUI_UltimateSDUpscale | Tile-based upscale |
| ComfyUI-TeaCache | Cache hızlandırma |
| ComfyUI-Crystools | Primitive helpers |
| ComfyUI-Image-Saver | Sampler selector |
| ComfyUI_LayerStyle | SAM segmentation |
| comfyui_memory_cleanup | VRAM temizleme |

---
# A) Kurulum + Custom Node'lar

In [ ]:
import os
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok — gated modeller indirilemez!')

# ComfyUI
COMFY_DIR = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFY_DIR}/custom_nodes'

if not os.path.exists(COMFY_DIR):
    print('\U0001f4e6 ComfyUI...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
    !pip install -q -r {COMFY_DIR}/requirements.txt
else:
    print('\u2705 ComfyUI mevcut')

!pip install -q -U --pre comfyui-manager

# Custom Node'lar
NODES = {
    'ComfyUI-Impact-Pack': 'https://github.com/ltdrdata/ComfyUI-Impact-Pack.git',
    'ComfyUI-Impact-Subpack': 'https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git',
    'rgthree-comfy': 'https://github.com/rgthree/rgthree-comfy.git',
    'ComfyUI-KJNodes': 'https://github.com/kijai/ComfyUI-KJNodes.git',
    'ComfyUI-DepthAnythingV2': 'https://github.com/kijai/ComfyUI-DepthAnythingV2.git',
    'ComfyUI_essentials': 'https://github.com/cubiq/ComfyUI_essentials.git',
    'ComfyUI_UltimateSDUpscale': 'https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git',
    'ComfyUI-TeaCache': 'https://github.com/welltop-cn/ComfyUI-TeaCache.git',
    'ComfyUI-Crystools': 'https://github.com/crystian/ComfyUI-Crystools.git',
    'ComfyUI-Image-Saver': 'https://github.com/alexopus/ComfyUI-Image-Saver.git',
    'ComfyUI_LayerStyle': 'https://github.com/chflame163/ComfyUI_LayerStyle.git',
    'comfyui_memory_cleanup': 'https://github.com/Gerschel/comfyui_memory_cleanup.git',
    'ComfyUI_Comfyroll_CustomNodes': 'https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes.git',
}

for name, url in NODES.items():
    node_dir = f'{CUSTOM_NODES}/{name}'
    if not os.path.exists(node_dir):
        print(f'  \u2193 {name}')
        !git clone --depth 1 {url} {node_dir}
        req_file = f'{node_dir}/requirements.txt'
        if os.path.exists(req_file):
            !pip install -q -r {req_file}
    else:
        print(f'  \u2713 {name}')

# Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('\n\u2705 Kurulum tamam')

---
# B) Model İndir

**Önemli:** FLUX.1 Dev/Fill/Depth gated model — önce HuggingFace'te kabul et:
- https://huggingface.co/black-forest-labs/FLUX.1-dev
- https://huggingface.co/black-forest-labs/FLUX.1-Fill-dev
- https://huggingface.co/black-forest-labs/FLUX.1-Depth-dev-lora

**CivitAI LoRA'ları** (manuel indirip `ComfyUI/models/loras/FLux_Lora/` altına koy):
- [chinfixer-2000](https://civitai.com/models/775002/chin-fixer-2000)
- [Pandora-RAWr (skin texture)](https://civitai.com/models/1688671)

In [ ]:
import shutil

from huggingface_hub import hf_hub_download

MODELS_DIR = f'{COMFY_DIR}/models'

def hf_download(repo, filename, dest_dir):
    """HuggingFace'ten dosya indir. Mevcutsa atla."""
    basename = filename.split('/')[-1]
    dest = f'{dest_dir}/{basename}'
    if os.path.exists(dest):
        print(f'  \u2713 {basename} (mevcut)')
        return
    print(f'  \u2193 {basename}...')
    os.makedirs(dest_dir, exist_ok=True)
    try:
        path = hf_hub_download(repo_id=repo, filename=filename, local_dir='/content/hf_cache')
        shutil.move(path, dest)
        print(f'  \u2705 {basename}')
    except Exception as e:
        print(f'  \u274c Ba\u015far\u0131s\u0131z: {e}')

# === Diffusion Models ===
print('\U0001f4e5 FLUX.1 Dev:')
hf_download(
    'black-forest-labs/FLUX.1-dev',
    'flux1-dev.safetensors',
    f'{MODELS_DIR}/diffusion_models'
)
# Workflow 'flux1-dev.sft' adıyla arıyor
src = f'{MODELS_DIR}/diffusion_models/flux1-dev.safetensors'
link = f'{MODELS_DIR}/diffusion_models/flux1-dev.sft'
if os.path.exists(src) and not os.path.exists(link):
    os.symlink(src, link)
    print('  \u2705 symlink: flux1-dev.sft -> flux1-dev.safetensors')

print('\n\U0001f4e5 FLUX.1 Fill Dev:')
hf_download(
    'black-forest-labs/FLUX.1-Fill-dev',
    'flux1-fill-dev.safetensors',
    f'{MODELS_DIR}/diffusion_models'
)

# === Text Encoders ===
print('\n\U0001f4e5 Text Encoders:')
hf_download(
    'comfyanonymous/flux_text_encoders',
    't5xxl_fp16.safetensors',
    f'{MODELS_DIR}/text_encoders'
)
hf_download(
    'comfyanonymous/flux_text_encoders',
    'clip_l.safetensors',
    f'{MODELS_DIR}/text_encoders'
)

# === VAE ===
print('\n\U0001f4e5 VAE:')
hf_download(
    'black-forest-labs/FLUX.1-dev',
    'ae.safetensors',
    f'{MODELS_DIR}/vae'
)

# === LoRA'lar ===
print('\n\U0001f4e5 LoRA\'lar:')
# Depth LoRA (gated)
hf_download(
    'black-forest-labs/FLUX.1-Depth-dev-lora',
    'flux1-depth-dev-lora.safetensors',
    f'{MODELS_DIR}/loras'
)
# CivitAI LoRA'ları için klasör
os.makedirs(f'{MODELS_DIR}/loras/FLux_Lora', exist_ok=True)
print('  \u26a0\ufe0f chinfixer-2000.safetensors -> CivitAI\'dan manuel indir')
print('  \u26a0\ufe0f Pandora-RAWr.safetensors -> CivitAI\'dan manuel indir')
print(f'  Konum: {MODELS_DIR}/loras/FLux_Lora/')

# === Preprocessor / Upscale / Detection Modelleri ===
print('\n\U0001f4e5 Depth Anything V2:')
os.makedirs(f'{MODELS_DIR}/depthanything', exist_ok=True)
hf_download(
    'Kijai/DepthAnythingV2-safetensors',
    'depth_anything_v2_vits_fp16.safetensors',
    f'{MODELS_DIR}/depthanything'
)

print('\n\U0001f4e5 SAM (Segment Anything):')
os.makedirs(f'{MODELS_DIR}/sams', exist_ok=True)
hf_download(
    'spaces/abhishek/StableSAM',
    'sam_vit_b_01ec64.pth',
    f'{MODELS_DIR}/sams'
)

print('\n\U0001f4e5 Ultralytics (Eyes Detection):')
os.makedirs(f'{MODELS_DIR}/ultralytics/bbox', exist_ok=True)
hf_download(
    'Tenofas/ComfyUI',
    'ultralytics/bbox/Eyeful_v2-Paired.pt',
    f'{MODELS_DIR}/ultralytics/bbox'
)

print('\n\U0001f4e5 Upscale Model:')
hf_download(
    'Phips/4xRealWebPhoto_v4_dat2',
    '4xRealWebPhoto_v4_dat2.pth',
    f'{MODELS_DIR}/upscale_models'
)

print('\n\u2705 Model indirme tamam')
print('\n\u26a0\ufe0f Eksik: CivitAI LoRA\'larını manuel indir ve FLux_Lora/ altına koy')

---
# C) ComfyUI Başlat

`USE_CLOUDFLARE = False` (default): Colab proxy

`USE_CLOUDFLARE = True`: Cloudflare tunnel — `comfyui.ersamely.com`

## Workflow Yükleme
1. ComfyUI açılınca **Load** → `Consistent Face 3x3 Generator.json`
2. 3x3 grid referans görsel yükle
3. Prompt'ta yüz detaylarını yaz
4. **Queue Prompt** → 9 tutarlı yüz görseli üretir (uzun sürer, normal)

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata, output

USE_CLOUDFLARE = False

PORT = 8188

subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

log_file = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', str(PORT), '--gpu-only', '--enable-cors-header', '*', '--enable-manager'],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)
print(f'\U0001f680 ComfyUI başlatıldı (PID: {comfy_proc.pid})')

t0 = time.time()
ready = False
while time.time() - t0 < 120:
    try:
        if requests.get(f'http://localhost:{PORT}/system_stats', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if comfy_proc.poll() is not None:
        print('\u274c ComfyUI \u00e7\u00f6kt\u00fc!')
        log_file.close()
        with open('/content/comfyui.log') as f:
            print(f.read()[-500:])
        break
    time.sleep(3)

if ready:
    print(f'\u2705 ComfyUI hazır ({int(time.time()-t0)}s)')
    if USE_CLOUDFLARE:
        token = userdata.get('CF_TUNNEL_TOKEN')
        cf_log = open('/content/cloudflared.log', 'w')
        cf_proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
            stdout=cf_log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        )
        time.sleep(5)
        print(f'\U0001f310 Cloudflare: https://comfyui.ersamely.com')
    else:
        print(f'\U0001f310 Colab Proxy:')
        output.serve_kernel_port_as_window(PORT, path='/')
else:
    print('\u274c Timeout!')

In [ ]:
import time
from datetime import datetime, timezone

import requests

print('ComfyUI canlı tutma. Durdurmak için interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'http://localhost:{PORT}/system_stats', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False
    comfy_alive = comfy_proc.poll() is None
    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    c = '\u2705' if (local_ok and comfy_alive) else '\u274c'
    print(f'{now} | ComfyUI: {c}')
    time.sleep(30)